# Suite VAL — Validators

`Validators.Validate(store, constraints)` evaluates each object's constraints, those of its own schema, and reports
what does not hold, what is unknown and what raised, labelled as mbse-schemas labels objects.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Constraints as C, Validators as V } from "@mbse/patterns";
import { Errors } from "@mbse/schemas/Framework";
import { Contact, Phone, assert, book, raises, same } from "./support.js";

const self = E.variable("this");
const constraints = new C.Set([
  new C.Constraint("Contact", "adult", self.age.ge(18n)),
  new C.Constraint("Contact", "phoned", E.operation("count", E.operation("entries", self, "phones")).ge(1n)),
  new C.Constraint("Phone", "numbered", self.has("number")),
]);
const store: any = book();
const ann = store.Contact().name("ann").age(30n).phones((e: any) => e.phone((p: any) => p.number("555"))).create();
const bob = store.Contact().name("bob").phones((e: any) => e.phone((p: any) => p)).create();

## VAL-01 · Each object is checked against its schema's constraints

In [ ]:
{
  const validate = V.Validate(store, constraints);
  assert(validate(Contact, ann).length === 0);
  assert(same(validate(Contact, bob), ["Contact#0: 'adult' is unknown"])); // bob's age is absent
  const young = store.Contact().name("cy").age(9n).create();
  assert(same(validate(Contact, young), ["Contact#0: 'adult' does not hold", "Contact#0: 'phoned' does not hold"]));
  assert(same(validate(Phone, store.Phone().create()), ["Phone#0: 'numbered' does not hold"]));
  assert(same(validate(Contact, store.Phone().create()), ["the value is a 'Phone', not an instance of the given schema"]));
}

## VAL-02 · Reachable checks every object reachable from the root

In [ ]:
{
  assert(same(V.Validate(store, constraints).Reachable(Contact, bob), ["Contact#0: 'adult' is unknown",
    "Phone#1: 'numbered' does not hold"]));
  assert(V.Validate(store, constraints).Reachable(Contact, ann).length === 0);
}

## VAL-03 · An unknown result is reported, ignored or a violation

In [ ]:
{
  assert(V.Validate(store, constraints, { unknown: "ignore" })(Contact, bob).length === 0);
  assert(same(V.Validate(store, constraints, { unknown: "violation" })(Contact, bob), ["Contact#0: 'adult' does not hold"]));
  assert(same(V.Validate(store, constraints, { unknown: "report" })(Contact, bob), ["Contact#0: 'adult' is unknown"]));
  raises(Errors.ValueError, () => V.Validate(store, constraints, { unknown: "maybe" }),
    "unknown must be 'report', 'ignore' or 'violation', got 'maybe'");
  assert(same(V.UNKNOWN, ["report", "ignore", "violation"]));
}

## VAL-04 · A rule that raises is reported in place; a rule that cannot be right is refused

In [ ]:
{
  const raising = new C.Set([new C.Constraint("Contact", "mixed", E.operation("add", self.age, 1.5).ge(2.0)),
    new C.Constraint("Contact", "number", self.age)]);
  assert(same(V.Validate(store, raising)(Contact, ann), [
    "Contact#0: 'mixed' raised TypeError: add expects numbers of one domain, got int and float",
    "Contact#0: 'number' raised TypeError: a predicate must be a bool, got int"]));
  raises(Errors.ValueError, () => V.Validate(store, [new C.Constraint("Contact", "x", E.variable("n"))]),
    "constraint 'x' of 'Contact': variable 'n' is not bound");
}